# Retrieval Augmented Generation (RAG)

## Task Description
Perform Retrieval Augmented Generation (RAG) to answer questions based on a specific document or context.

**Focus:** This notebook is optimized for **Google Colab Free Tier** (T4 GPU). We will use `LangChain` for the RAG pipeline and a 4-bit quantized LLM (`Zephyr-7B`).

In this notebook, we will:
1. **Install & Import:** Set up `langchain`, `faiss-cpu`, `sentence-transformers`, and `bitsandbytes`.
2. **Load Data:** Create or load a knowledge base (documents).
3. **Embeddings:** Convert text into vector embeddings using `sentence-transformers`.
4. **Vector Store:** Index the embeddings using `FAISS` for fast retrieval.
5. **LLM Setup:** Load `Zephyr-7B-beta` in 4-bit precision.
6. **RAG Pipeline:** Combine the retriever and the LLM to answer questions.

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source models and is optimized to run on the Google Colab Free Tier.

## 1. Installation
We need `langchain` for the framework, `faiss-cpu` for the vector database, and `sentence-transformers` for embeddings.

In [ ]:
!pip install matplotlib -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers torch accelerate bitsandbytes

## 2. Imports and Setup
Import necessary libraries.

In [ ]:
import torch
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 3. Load Data
Load knowledge base.

In [ ]:
# Sample Knowledge Base
text_data = """
Artificial Intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans.
The history of AI began in antiquity, with myths, stories and rumors of artificial beings endowed with intelligence or consciousness by master craftsmen.
The field of AI research was founded as an academic discipline in 1956.
The Dartmouth Conference of 1956 is considered the birth of AI as a field.
In the twenty-first century, AI techniques have experienced a resurgence following concurrent advances in computer power, large amounts of data, and theoretical understanding.
Deep learning, a subset of machine learning, has enabled significant progress in image and speech recognition.
Generative AI, such as ChatGPT and Stable Diffusion, can create new content, including text, images, audio, and video.
"""

# Text Splitting (Chunking)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)
docs = text_splitter.create_documents([text_data])

print(f"Split data into {len(docs)} chunks.")
print("Example chunk:", docs[0].page_content)

## 4. Data Cleaning (Embeddings)
Prepare vectors.

In [ ]:
# Load Embedding Model
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name, model_kwargs={'device': device})

# Create Vector Store
vector_store = FAISS.from_documents(docs, embeddings)
print("Vector store created.")

## 5. Load Model
Load the LLM.

In [ ]:
model_id = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Create Generation Pipeline
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    repetition_penalty=1.15
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
print("LLM Pipeline ready.")

## 6. Inference
Run RAG.

In [ ]:
# Setup Retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2}) # Retrieve top 2 chunks

# Create QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# Query
query = "When was the field of AI founded?"
result = qa_chain.invoke({"query": query})

print(f"\nQuery: {query}")
print(f"Answer: {result['result']}")
print("\nSource Documents:")
for doc in result['source_documents']:
    print(f"- {doc.page_content}")

## 7. Save Results
Save output.

In [ ]:
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'frames' in locals():
        f.write("Video generated.")
    elif 'image' in locals():
        f.write("Image generated.")
    elif 'text' in locals():
        f.write(str(text))
    else:
        f.write("Task completed.")
print('Results saved to results.txt')


## 7. Advanced Usage
Try using a different model or your own data.


In [ ]:
# Example: Load a different model
# model_name = "..."
# classifier = pipeline(..., model=model_name)
# results = classifier(your_input)
print("Try modifying the code above to use a different model!")